# M10 - Embeddings, retrieval, and retrieval-augmented generation

**Level 2 · CPU-only · about 90 minutes · no API key required**

This notebook turns retrieval into an answer with evidence. It builds a small
index, retrieves passages, calls a deterministic offline model, validates the
citations, and demonstrates abstention when the corpus does not support the
question. The model is a stand-in; the pipeline boundaries are the lesson.

## Learning objectives

By the end of this notebook you can:

1. build a retrieval index over cited chunks;
2. assemble a grounded prompt and call a model through one interface;
3. validate citation numbers against the retrieved registry;
4. force an abstention and explain why a similarity score is not evidence;
5. identify the retrieval and generation failure separately;
6. design a fixed evaluation set with answerable and unanswerable questions.

## Prerequisites

- M09, or equivalent familiarity with chunks, BM25, and cosine similarity.
- Python functions and dictionaries.
- CPU only. The offline client is extractive, deterministic, and not a language
  model; it is used so the pipeline can be tested without downloads.

## Conceptual explanation

Retrieval-augmented generation has four contracts:

1. **Evidence contract** — the retrieved chunks are the only allowed facts.
2. **Citation contract** — every factual sentence points to a retrieved chunk.
3. **Abstention contract** — no supporting evidence means no answer.
4. **Evaluation contract** — a fixed question set and a gold document id.

If any contract is implicit, a fluent answer can hide a broken system. The
pipeline in this notebook makes each contract a separate function so it can fail
visibly.

## Important terminology

| Term | Meaning |
| --- | --- |
| RAG | Retrieval-augmented generation |
| Grounding | Answering only from retrieved evidence |
| Citation precision | Fraction of cited sources that are in the gold set |
| Citation recall | Fraction of gold sources that were cited |
| Abstention | Refusing when evidence is missing or insufficient |
| Hallucinated citation | A citation number that was never retrieved |
| Hit@k | Whether the gold document appears in the top *k* retrieved chunks |


## 1. Set up the repository imports


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parents[1]
PROJECT = ROOT / "projects" / "chatbots-rag"
sys.path.insert(0, str(PROJECT))
print("Repository root:", ROOT.name)
print("Section package:", PROJECT.name)


Repository root: AITopics
Section package: chatbots-rag


## 2. Build an evidence corpus

The corpus deliberately includes two documents that disagree about a deadline.
That is not an accident: conflict detection is part of the grounding contract.
The system should surface the disagreement rather than average it away.


In [2]:
from chatbot_core import Chunk

DOCS = [
    ("registration", "Registration", "Deadline",
     "Registration for the autumn cohort closes on 15 September. Late registration is not available."),
    ("registration-old", "Registration archive", "Deadline",
     "The previous year's registration closed on 30 September. This archive is retained for reference only."),
    ("hardware", "Hardware guide", "Memory",
     "The CPU path needs at least 8 GB of RAM. A GPU is optional for the foundation modules."),
    ("assessment", "Assessment policy", "Weights",
     "The guided notebook is worth 30 percent and the independent lab is worth 40 percent."),
    ("support", "Support", "Response time",
     "The course forum is monitored on weekdays. Include the module code and the full error message."),
]

chunks = []
for doc_id, title, section, text in DOCS:
    chunks.append(
        Chunk(
            chunk_id=f"{doc_id}-1",
            doc_id=doc_id,
            title=title,
            section=section,
            text=text,
            source=f"{doc_id}.md",
            start=0,
            end=len(text),
        )
    )

print("documents:", len(chunks))
print("sources:", sorted({chunk.source for chunk in chunks}))


documents: 5
sources: ['assessment.md', 'hardware.md', 'registration-old.md', 'registration.md', 'support.md']


## 3. Retrieve evidence and inspect the ranked list

The retriever returns chunks, scores, and ranks. Keep all three. A citation needs
the chunk identity; the threshold needs the score; the evaluation needs the rank.


In [3]:
import pandas as pd
from chatbot_core import HashingEmbedder, HybridRetriever

retriever = HybridRetriever(chunks, HashingEmbedder(dim=256), vector_weight=0.6)
question = "When does registration close?"
results = retriever.search(question, k=3)

pd.DataFrame(
    [
        {
            "rank": result.rank,
            "score": round(result.score, 4),
            "source": result.chunk.source,
            "section": result.chunk.section,
            "text": result.chunk.text[:100] + "...",
        }
        for result in results
    ]
)


,rank,score,source,section,text
0,1,1.0000,registration.md,Deadline,Registration for the autumn cohort closes on 1...
1,2,0.7758,registration-old.md,Deadline,The previous year's registration closed on 30 ...
2,3,0.2406,hardware.md,Memory,The CPU path needs at least 8 GB of RAM. A GPU...


## 4. Generate a grounded answer

`FakeModelClient` is an extractive stand-in: it selects sentences with content
overlap and returns JSON with citations. It has no semantic understanding. Its
value is that the surrounding contracts can be tested deterministically. In a
real system, replace it with a local or hosted client behind the same interface.


In [4]:
from chatbot_core import FakeModelClient, GenerationConfig, generate, render_answer

client = FakeModelClient()
config = GenerationConfig(k=3, min_score=0.01, require_citations=True)
answer = generate(
    question=question,
    chunks=[result.chunk for result in results],
    scores=[result.score for result in results],
    client=client,
    config=config,
)

print(render_answer(answer))
print()
print("abstained:", answer.abstained)
print("top score:", round(answer.top_score, 4))


Registration for the autumn cohort closes on 15 September. Late registration is not available.

Sources:
  [1] Registration - Deadline (registration.md)

abstained: False
top score: 1.0


## 5. Validate citations against the retrieved registry

A citation is valid only if its number exists in the registry created from the
same retrieved chunks. A fabricated `[99]` must fail closed: replace the answer
with a refusal, not a footnote disclaimer.


In [5]:
from chatbot_core import build_registry, render_answer, validate

registry = build_registry(
    [result.chunk for result in results],
    [result.score for result in results],
)

claimed = list(range(1, len(results) + 1))
outcome = validate(claimed, answer.text, registry)
print("valid:", outcome.ok)
print("valid citations:", [citation.display() for citation in outcome.valid])

fabricated = validate([99], "This claim cites [99].", registry)
print("fabricated accepted:", fabricated.ok)
print("invalid numbers:", fabricated.invalid_numbers)
print("fabricated markers:", fabricated.fabricated_in_text)


valid: True
valid citations: ['[1] Registration - Deadline (registration.md)', '[2] Registration archive - Deadline (registration-old.md)', '[3] Hardware guide - Memory (hardware.md)']
fabricated accepted: False
invalid numbers: [99]
fabricated markers: [99]


## 6. Force abstention with a score gate

Abstention should happen before a model is asked to answer. Here the threshold is
raised above any possible score, so the pipeline refuses without calling the
client. The refusal is a policy decision, not an admission that the model is
broken.


In [6]:
strict = GenerationConfig(k=3, min_score=0.99)
refusal = generate(
    question="Who won the 2022 World Cup?",
    chunks=[result.chunk for result in results],
    scores=[result.score for result in results],
    client=client,
    config=strict,
)
print(render_answer(refusal))
print()
print("abstained:", refusal.abstained)


I do not have enough evidence in the provided documents to answer that.

No sources cited: the documents do not support an answer.

abstained: True


## 7. Retrieval and generation are evaluated separately

For each question, record the gold document, whether it appeared in the top *k*,
whether the answer abstained, and whether its cited sources overlap the gold set.
Do not collapse those columns into one score. A generation fix cannot repair a
retrieval miss, and a retrieval fix cannot make an unsupported claim true.


In [7]:
questions = [
    {
        "id": "q1",
        "question": "When does registration close?",
        "gold": ["registration.md"],
        "answerable": True,
    },
    {
        "id": "q2",
        "question": "How much RAM does the CPU path need?",
        "gold": ["hardware.md"],
        "answerable": True,
    },
    {
        "id": "q3",
        "question": "What is the capital of Portugal?",
        "gold": [],
        "answerable": False,
    },
]

rows = []
for item in questions:
    found = retriever.search(item["question"], k=3)
    docs = [result.chunk.source for result in found]
    hit = any(gold in docs for gold in item["gold"]) if item["gold"] else True
    rows.append(
        {
            "id": item["id"],
            "answerable": item["answerable"],
            "gold": ", ".join(item["gold"]) or "(none)",
            "hit@3": hit,
            "retrieved": ", ".join(docs),
        }
    )

pd.DataFrame(rows)


,id,answerable,gold,hit@3,retrieved
0,q1,True,registration.md,True,"registration.md, registration-old.md, hardware.md"
1,q2,True,hardware.md,True,"hardware.md, assessment.md, support.md"
2,q3,False,(none),True,"hardware.md, assessment.md, support.md"


## Explanation of the code

- `HybridRetriever` is built once from the corpus and embedder; the index is the
  artifact reused for every query.
- `generate` applies the evidence gate before calling the client and returns an
  `Answer` object carrying text, citations, abstention, and the top score.
- `build_registry` numbers the retrieved chunks exactly as the prompt does.
- `validate` checks both structured citation numbers and inline `[n]` markers.
- The evaluation loop records retrieval and refusal separately; it does not use a
  model-graded "quality" score.

## Expected output

You should see a ranked retrieval table, an answer with a cited source, `valid:
True` for the legitimate citation set, `fabricated accepted: False` for `[99]`,
and an abstention for the unsupported question. The exact answer text is produced
by the extractive client and will change if the corpus changes.

## Interpretation of results

The valid-citation check proves only that the cited chunk was retrieved. It does
not prove the citation supports the sentence. Citation *support* requires a
separate check or human review. The evaluation table shows the same distinction:
`hit@3` is a retrieval metric, while abstention is a generation-policy outcome.


## Expected output

You should see five documents with source names, a ranked table for the registration
question, a grounded answer with numbered citations, `valid: True` for the cited
registry, and `fabricated accepted: False` with `invalid numbers: [99]`. The strict
configuration should return the refusal text with `abstained: True`. The evaluation
table should show `hit@3` for the two answerable questions and a refusal path for
the unanswerable one.

## Interpretation of results

The important distinction is between citation validity and factual support. A
citation number can resolve to a real retrieved chunk and still not support the
sentence that cites it; this notebook validates the first property and asks you to
check the second. The registration answer also shows a conflict signal: the current
policy and the archived policy state different deadlines, so a useful system must
name the conflict instead of silently choosing one. Abstention is a successful
outcome when the evidence is absent, and it must be measured separately from
retrieval quality.

## Practical exercises

1. Add a second hardware document and see whether the answer cites both sources.
2. Replace the question with one whose answer is only implied, not stated. Does
   the system abstain or answer from a nearby sentence? Explain the difference.
3. Add a poison document containing an instruction to ignore the system prompt.
   Verify that `generate` quarantines it instead of using it as evidence.
4. Compute citation precision and recall by hand for one answer and compare with
   the definitions in the terminology table.

## Challenge

Create a 12-question evaluation set with six answerable and six unanswerable
questions. Report hit@3, abstention accuracy, and citation precision. Then write
one paragraph explaining which failure dominates and what you would change first.

## Limitations

The default `FakeModelClient` is an extractive stand-in, not a language model. It
selects a sentence with lexical overlap and can miss a correct passage phrased in
different words. The corpus is small and synthetic, the score gate is tuned to
this demonstration, and citation validation proves only that a citation exists
in the retrieved set, not that the sentence entails the claim. Validate retrieval
and generation separately, as this notebook does, before considering a hosted
model.

## Common mistakes and troubleshooting

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| Answer always abstains | Threshold too high or query has no lexical overlap | Inspect `top_score` and the retrieved text |
| Citation validation fails | Registry built from different chunks than the prompt | Build the registry from the exact retrieved list |
| Answer cites a real number but wrong source | Citation number is valid but unsupported | Add support checking; do not confuse validity with entailment |
| Evaluation always passes | Gold ids and retrieved ids use different source names | Print both lists and normalize the source identifiers |
| Conflict not detected | The conflicting values use different units or documents | Add a test case with the same unit and two documents |

## Summary

You implemented a complete RAG path: cited chunks, hybrid retrieval, grounded
generation, citation validation, abstention, and separate retrieval/generation
evaluation. The offline client is intentionally simple; the contracts around it
are the reusable engineering.

## Further reading

- Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks", NeurIPS 2020.
- Karpukhin et al., "Dense Passage Retrieval for Open-Domain Question Answering", EMNLP 2020.
- The repository's `projects/chatbots-rag/` projects and `tests/test_chatbot_rag.py`.
